#  Figures (from saved results only)



In [1]:
import os, sys, torch
import warnings
warnings.filterwarnings("ignore")   # suppress any path-related warnings

sys.path.insert(0, os.path.abspath('.'))

from src.config import RESULTS_DIR, CHECKPOINT_DIR, PLOTS_DIR
from src import evaluate
from src import probabilistic_fusion as pf

FINAL_DIR = os.path.join(PLOTS_DIR, "final")
os.makedirs(FINAL_DIR, exist_ok=True)
print("Final figures directory ready.")

[config] Using device: mps
Final figures directory ready.


## 1. Table II comparison bar chart 

In [2]:
_results_path = os.path.join(RESULTS_DIR, "results_final.pt")
if not os.path.exists(_results_path):
    raise FileNotFoundError(
        f"{_results_path} not found -- run nb1's final evaluation cell "
        f"(and nb4's PatchCleanser-merge cell) first to produce results_final.pt."
    )

results_final = torch.load(_results_path, weights_only=False)
evaluate.print_results_with_ci(results_final)

# Re-generate just the bar chart (radar already skipped inside generate_all_plots)
evaluate.generate_all_plots(results_final)

import shutil
src_path = os.path.join(PLOTS_DIR, "results_table_with_ci.png")
dst_path = os.path.join(FINAL_DIR, "table2_comparison_final.png")
shutil.copy(src_path, dst_path)
print("Table 2 comparison figure saved.")


Method                       Clean acc       Attack SR       Recovered          FPR
-------------------------------------------------------------------------------------
no_defense             99.8% [99.5-100.0] 84.3% [82.1-86.4] 15.7% [13.6-17.9] 0.0% [0.0-0.0]
jpeg                   99.8% [99.5-100.0] 46.1% [43.0-49.4] 53.9% [50.6-57.0] 0.2% [0.0-0.5]
feature_squeeze        90.6% [88.8-92.3] 21.2% [18.7-23.7] 78.8% [76.3-81.3] 9.4% [7.7-11.2]
gradcam_only           96.5% [95.4-97.6]  2.3% [1.4-3.2] 97.7% [96.8-98.6] 3.5% [2.4-4.8]
hybrid ☆               99.5% [99.0-99.9] 15.3% [13.2-17.6] 84.7% [82.4-86.8] 0.5% [0.1-1.0]
pmeapi ★               99.8% [99.5-100.0]  7.4% [5.9-8.9] 92.6% [91.1-94.1] 0.2% [0.0-0.5]
patchcleanser          99.6% [99.2-99.9] 34.9% [31.8-37.8] 65.1% [62.0-68.1] 0.4% [0.1-0.8]
Note: 95% bootstrap confidence intervals in brackets

Results with CI plot saved.
Attack vs defense plot saved.
Radar chart saved.
GradCAM grid saved.
All plots saved.
Table 2 compariso

## 2. Sensor calibration figure (real fitted Beta parameters + real empirical scores)

In [3]:
_params_path = os.path.join(CHECKPOINT_DIR, "pmeapi_params.pt")
if not os.path.exists(_params_path):
    raise FileNotFoundError(
        f"{_params_path} not found -- run nb4's calibrate() cell first."
    )

params = pf.load_calibration()
if "_raw_scores" not in params:
    raise KeyError(
        "pmeapi_params.pt does not contain '_raw_scores'"
    )

pf._plot_calibration(params, params["_raw_scores"])

import shutil
src_path = os.path.join(PLOTS_DIR, "pmeapi_calibration.png")
dst_path = os.path.join(FINAL_DIR, "pmeapi_calibration_final.png")
shutil.copy(src_path, dst_path)
print("PMEAPI calibration figure saved.")

Calibration plot saved.
PMEAPI calibration figure saved.


## 3. Decision-threshold analysis (real computed posteriors)


In [4]:
_post_path = os.path.join(CHECKPOINT_DIR, "pmeapi_threshold_posteriors.pt")
if not os.path.exists(_post_path):
    raise FileNotFoundError(
        f"{_post_path} not found -- re-run nb4's tune_decision_threshold() "
        f"cell once (it now saves this file automatically as part of the fix)."
    )

posteriors = torch.load(_post_path, weights_only=False)
print(f"Loaded real posteriors: N={posteriors['n_images']}  "
      f"threshold={posteriors['best_thresh']:.3f}  "
      f"detection={100*posteriors['best_det']:.1f}%  "
      f"target_fpr={100*posteriors['target_fpr']:.1f}%")

save_path = os.path.join(FINAL_DIR, "pmeapi_threshold_analysis_final.png")
pf.plot_threshold_analysis(
    posteriors["probs_clean"],
    posteriors["probs_patched"],
    posteriors["best_thresh"],
    posteriors["target_fpr"],
    save_path=save_path,
)
print("Threshold analysis figure saved.")

Loaded real posteriors: N=250  threshold=0.991  detection=74.4%  target_fpr=1.0%
Threshold analysis plot saved.
Threshold analysis figure saved.


## 4. Adaptive patch training losses


In [5]:
import shutil
_losses_path = os.path.join(PLOTS_DIR, "adaptive_patch_losses.png")
if not os.path.exists(_losses_path):
    raise FileNotFoundError(
        f"{_losses_path} not found -- run nb2's adaptive-patch training cell first."
    )
dst_path = os.path.join(FINAL_DIR, "adaptive_patch_losses_final.png")
shutil.copy(_losses_path, dst_path)
print("Adaptive patch losses figure saved.")

Adaptive patch losses figure saved.
